In [ ]:
import pandas as pd
import glob
import os
import re

# Caminho da pasta onde estão os dados meteorológicos já extraídos
caminho_pasta = "../data/dados_metereologicos/dados_extraidos"

# Busca apenas os dados de estações do Rio Grande do Sul (RS)
arquivos = glob.glob(os.path.join(caminho_pasta, 'INMET_S_RS_*.csv'))

lista_df = []

for arquivo in arquivos:
    try:
        
        nome_arquivo = os.path.basename(arquivo)

        # Extraí o nome da cidade do nome do arquivo para adicionar como coluna posteriormente
        match = re.search(r'INMET_S_RS_[^_]+_(.*?)_\d{2}-\d{2}-\d{4}', nome_arquivo)
        cidade = match.group(1).strip().upper() if match else 'DESCONHECIDO'

        df = pd.read_csv(arquivo, sep=';', encoding='latin1', skiprows=8)

        df.columns = [col.strip() for col in df.columns]

        colunas_data = [col for col in df.columns if 'data' in col.lower()]
        
        if colunas_data:
            df['data'] = pd.to_datetime(df[colunas_data[0]], errors='coerce')
        else:
            df['data'] = pd.NaT

        df = df[df['data'].notna()]

        # Seleção das colunas relevantes para o estudo
        colunas_mapeamento = {
            'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao',
            'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temperatura',
            'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade',
            'VENTO, VELOCIDADE HORARIA (m/s)': 'vento'
        }

        colunas_presentes = {k: v for k, v in colunas_mapeamento.items() if k in df.columns}

        df = df[list(colunas_presentes.keys()) + ['data']]
        df = df.rename(columns=colunas_presentes)

        df['cidade'] = cidade


        lista_df.append(df)

    except Exception as e:
        print(f"Erro no arquivo {arquivo}: {e}")

# Concatenar
df_final = pd.concat(lista_df, ignore_index=True)


✅ Dados tratados e salvos com sucesso!


In [12]:
df_final

,precipitacao,temperatura,umidade,vento,data,cidade
0,0.0,25.0,72.0,NaN,2022-01-01,PORTO ALEGRE - JARDIM BOTANICO
1,0.0,NaN,77.0,NaN,2022-01-01,PORTO ALEGRE - JARDIM BOTANICO
2,0.0,NaN,80.0,1.0,2022-01-01,PORTO ALEGRE - JARDIM BOTANICO
3,0.0,NaN,79.0,1.0,2022-01-01,PORTO ALEGRE - JARDIM BOTANICO
4,0.0,23.0,84.0,NaN,2022-01-01,PORTO ALEGRE - JARDIM BOTANICO
...,...,...,...,...,...,...
4079155,0.0,NaN,28.0,NaN,2022-12-31,PORTO ALEGRE- BELEM NOVO
4079156,0.0,NaN,35.0,NaN,2022-12-31,PORTO ALEGRE- BELEM NOVO
4079157,0.0,NaN,48.0,NaN,2022-12-31,PORTO ALEGRE- BELEM NOVO
4079158,0.0,NaN,52.0,NaN,2022-12-31,PORTO ALEGRE- BELEM NOVO


In [15]:
# Limpeza dos dados
df_final.replace([-9999, '-9999', ''], pd.NA, inplace=True)

for col in ['precipitacao', 'temperatura', 'umidade', 'vento']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')


# Agregação diária por cidade
df_diario = df_final.groupby(['cidade', 'data']).agg({
    'precipitacao': 'sum',
    'temperatura': 'mean',
    'umidade': 'mean',
    'vento': 'mean'
}).reset_index()

# Criar coluna de ano
df_diario['ano'] = df_diario['data'].dt.year

# Criar coluna de bairro
df_diario['bairro'] = df_diario['cidade'].str.split(' - ').str[-1]

df_diario['cidade'] = df_diario['cidade'].str.split(' - ').str[0]

# Salvar arquivo tratado
df_diario.to_csv('..\data\dados_metereologicos\dados_rs_tratados\dados_meteorologia_RS.csv', index=False)

print(" Dados tratados e salvos com sucesso!")

<>:26: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:26: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Bruno\AppData\Local\Temp\ipykernel_29108\3048373394.py:26: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df_diario.to_csv('..\data\dados_metereologicos\dados_rs_tratados\dados_meteorologia_RS.csv', index=False)


 Dados tratados e salvos com sucesso!
